# Algoritmo Genético para la Resolución de Laberintos: Aceleración en GPU (NVIDIA CUDA)

Este cuaderno repasa la arquitectura de hardware, la paralelización física mediante hilos CUDA y la implementación del motor acelerado en nuestra GPU NVIDIA GTX 1650.

---

## 1. El Cuello de Botella de Python (GIL) y la Solución JIT

### El Problema de Python y el GIL
Los bucles tradicionales en Python (`for` o `while`) son notoriamente lentos al procesar millones de celdas debido a:
1. **Interpretación línea a línea**: Python es un lenguaje dinámico interpretado; cada instrucción de bucle se evalúa dinámicamente en tiempo de ejecución, lo que añade un enorme overhead de CPU.
2. **GIL (Global Interpreter Lock)**: El GIL de Python restringe la ejecución de código Python a un único hilo de CPU de manera simultánea. Esto significa que agregar hilos convencionales de Python para paralelizar la simulación sobre la CPU no produce ninguna ganancia real en procesamiento matemático.

### Compilación JIT de Numba
Numba soluciona este cuello de botella mediante **Compilación Just-In-Time (JIT)**. Traduce dinámicamente las funciones seleccionadas de Python a código de máquina optimizado utilizando la infraestructura de compiladores **LLVM**. 
Cuando usamos `@cuda.jit`, Numba traduce directamente el kernel Python a instrucciones **PTX (Parallel Thread Execution)**, que es el ensamblador nativo de bajo nivel que se ejecuta de forma directa sobre los núcleos de hardware de las tarjetas gráficas NVIDIA.

## 2. Arquitectura Masiva Paralela: CPU vs. GPU

| Característica | CPU (Central Processing Unit) | GPU (Graphics Processing Unit) |
|---|---|---|
| **Diseño** | Pocos núcleos físicos (ej. 4 u 8) diseñados para latencia mínima. | Miles de núcleos pequeños diseñados para ancho de banda máximo. |
| **Optimización** | Optimizado para lógica secuencial compleja con predicción de saltos e hilos ultra rápidos. | Optimizado para ejecutar la misma instrucción sobre miles de datos simultáneos (SIMD/SIMT). |
| **Modelo** | Flujo secuencial de instrucciones. | Ejecución paralela de múltiples hilos en paralelo sobre bloques de memoria. |

En nuestro laberinto, simular una población de $N = 1001$ individuos requiere simular $1001$ trayectorias completamente **independientes**. En la CPU, esto se procesa una por una secuencialmente. En la GPU, cada uno de los 1001 individuos se simula en un núcleo físico diferente de forma **simultánea y en paralelo**, reduciendo drásticamente el tiempo de cómputo.

## 3. Física de Hilos en CUDA: Warps, Bloques y Grillas

### Jerarquía de Hilos CUDA
*   **Hilos (Threads)**: La unidad básica de ejecución que corre el kernel.
*   **Warps**: Los hilos físicos de NVIDIA se agrupan en conjuntos de **32 hilos** llamados *Warps*. Los 32 hilos de un Warp ejecutan exactamente la misma instrucción física al mismo tiempo.
*   **Bloques (Blocks)**: Colección de hilos. Elegimos `threadsperblock = 256` (que es un múltiplo exacto de 32), maximizando el porcentaje de ocupación de hardware (occupancy) en los multiprocesadores de la GTX 1650.
*   **Grilla (Grid)**: Conjunto de todos los bloques necesarios para cubrir la población $N$.

### Asignación de Grilla Dinámica
El número de bloques necesarios se calcula dinámicamente con:
$$blockspergrid = \left\lceil \frac{N}{threadsperblock} \right\rceil = \frac{N + (threadsperblock - 1)}{threadsperblock}$$

### Mapeo Lineal y Condición de Resguardo
La instrucción `i = cuda.grid(1)` calcula el índice lineal global del hilo actual en toda la grilla:
$$i = blockIdx.x \times blockDim.x + threadIdx.x$$

Dado que el número de hilos totales ($blockspergrid \times threadsperblock$) puede ser mayor que el tamaño de la población $N$, es vital agregar la **condición de resguardo** `if i < N:` para evitar que los hilos sobrantes accedan a rangos de memoria inválidos o provoquen fallas de segmentación en la memoria global de la GPU.

In [ ]:
# Kernel CUDA simplificado de aceleracion_gpu.py mostrando la estructura paralela
from numba import cuda
import numpy as np

@cuda.jit
def _kernel_simular_cuda(
    poblacion_num, mapa_num, inicio_r, inicio_c, meta_r, meta_c,
    N, n, filas, cols, distancia_final_arr, es_valido_arr
):
    # 1. Obtener el índice único de este hilo
    i = cuda.grid(1)
    
    # 2. Condición de resguardo para proteger la memoria
    if i < N:
        p_r = inicio_r
        p_c = inicio_c
        
        # Simulación paso a paso del cromosoma asignado al hilo i
        for k in range(1, n + 1):
            gen = poblacion_num[i, k - 1]
            # (Lógica del movimiento de la trayectoria del individuo...)
            
        # Almacenar los resultados finales de este individuo
        distancia_final_arr[i] = abs(p_r - meta_r) + abs(p_c - meta_c)

## 4. Ley de Amdahl y Overhead de Transferencia

### Ley de Amdahl
El factor de aceleración máximo teótico $S$ de un programa está limitado por la fracción secuencial del código (aquella que no se puede ejecutar en paralelo, como cargar el CSV, ordenar la población y graficar resultados):
$$S = \frac{1}{(1 - P) + \frac{P}{s_p}}$$

Donde $P$ es la fracción paralelizada (la simulación física de la trayectoria) y $s_p$ es la aceleración de esa fracción en la GPU.

### Overhead de Transferencia y PCIe
La GPU está conectada a la placa madre a través del bus **PCI Express (PCIe)**. Transferir datos entre la memoria RAM principal (Host) y la memoria de video VRAM de la GPU (Device) toma un tiempo físico no despreciable. 
Si transferimos datos desorganizados en memoria, la GPU tardará más tiempo en cargarlos. Por ello, es imperativo garantizar la **C-contigüedad** en la memoria del Host usando `np.ascontiguousarray` antes de hacer el envío con `cuda.to_device()` para que la transferencia se realice en un único bloque contiguo de bytes a máxima velocidad.

In [ ]:
# Fragmento de código de gestión Host-to-Device y Device-to-Host en aceleracion_gpu.py
def simular_poblacion_acelerada(poblacion, mapa, inicio, meta):
    N = len(poblacion)
    n = len(poblacion[0])
    
    # Asegurar arrays C-contiguos en Host
    poblacion_num = np.ascontiguousarray(np.zeros((N, n), dtype=np.int8))
    mapa_num = np.ascontiguousarray(mapa, dtype=np.int8)
    
    # Transferencia explícita Host-to-Device
    d_poblacion = cuda.to_device(poblacion_num)
    d_mapa = cuda.to_device(mapa_num)
    
    # Reservar outputs en la VRAM de la GPU
    d_distancia_final = cuda.device_array(N, dtype=np.int32)
    
    # (... Lanzar Kernel y sincronizar ...)
    
    # Recuperación Device-to-Host
    dist_list = d_distancia_final.copy_to_host().tolist()
    return dist_list

## 5. Medición de Tiempos y el Factor de Aceleración S

Para calcular las métricas del benchmark, medimos el tiempo total transcurrido durante la evolución usando `time.perf_counter()`. El **Speedup neto** $S$ se define por:
$$S = \frac{T_{sec}}{T_{acc}}$$

Donde:
*   $T_{sec}$: Tiempo total de la CPU secuencial corriendo el ciclo evolutivo.
*   $T_{acc}$: Tiempo total con aceleración GPU (incluyendo compilación previa y transferencias explícitas de VRAM).